# Fine-tune MMS (adapter-based) — matched to your za-african-next-voices workflow

Local RTX 3060 12GB + 32GB RAM. This mirrors the pipeline from your
`finetune_wav2vec2.ipynb`: same dataset source, same `transcript` field,
same normalization (diacritics preserved), same `is_valid_transcript` filter,
same modern `transformers` API (no deprecated `as_target_processor()`,
uses `processing_class=`).

**What's different from plain Wav2Vec2-XLSR here:**
- Backbone (`facebook/mms-1b-all`) is frozen; only a small per-language
  adapter (~2.5M params) + `lm_head` train. Far fewer parameters moving
  means it's naturally more resistant to the blank-collapse you hit in the
  XLSR run — there's much less capacity to overfit into a degenerate
  single-token solution early in training.
- `ctc_zero_infinity=True` and `max_grad_norm=1.0` are carried over from
  your fixes, plus a couple of extra collapse diagnostics inline so you can
  catch a collapse within the first few eval steps instead of after a full run.

**Before running full-scale:** this notebook defaults to the same small
`.select(range(...))` subsets you used for fast iteration. Bump those up
once the small-scale run looks healthy (real, non-degenerate predictions).

## 1. Install dependencies

In [ ]:
!pip install -q transformers datasets evaluate jiwer accelerate soundfile librosa torchcodec

## 2. Check GPU

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

## 3. Config — edit these

`TARGET_LANG` should be an ISO 639-3 code. Section 6 checks whether MMS already ships an adapter for it.

In [ ]:
MODEL_ID = "facebook/mms-1b-all"
LANGUAGE = "tsn"  # tsn=Setswana, nso=Sepedi, ven=Tshivenda — matches your dataset config name
OUTPUT_DIR = f"./mms-{LANGUAGE}"

# RTX 3060 12GB, adapter-only training: backbone frozen, so headroom is
# better than full fine-tuning of a 1B model — but keep these modest while
# iterating on small subsets, same as your wav2vec2 notebook.
PER_DEVICE_TRAIN_BATCH = 2
PER_DEVICE_EVAL_BATCH = 2
GRAD_ACCUM_STEPS = 8

CHARS_TO_IGNORE = r'[,\?\.!\-\;\:"“%‘”�0-9\[\]\'\_]'

## 4. Imports

In [ ]:
import re
import json
import numpy as np
import torch
from dataclasses import dataclass
from typing import Dict, List, Union
from datasets import load_dataset, Audio, Dataset
from transformers import (
    Wav2Vec2CTCTokenizer,
    Wav2Vec2FeatureExtractor,
    Wav2Vec2Processor,
    Wav2Vec2ForCTC,
    AutoProcessor,
    TrainingArguments,
    Trainer,
)
import evaluate

## 5. Load and normalize data

Same source and split structure as your wav2vec2 notebook. Small `.select()` subsets for fast iteration — widen once things look healthy.

In [ ]:
dataset_dict = load_dataset(
    "dsfsi-anv/za-african-next-voices-compressed",
    LANGUAGE,
)

dataset_dict["train"] = dataset_dict["train"].select(range(200))
dataset_dict["dev_test"] = dataset_dict["dev_test"].select(range(200))
dataset_dict["dev"] = dataset_dict["dev"].select(range(10))

dataset_dict = dataset_dict.cast_column("audio", Audio(sampling_rate=16000))

In [ ]:
def normalize_text(batch):
    if batch["transcript"] is None:
        return batch

    text = batch["transcript"]

    # Remove annotation tags like [pause], [cs], [?], [noise] etc.
    text = re.sub(r'\[.*?\]', '', text)

    # Lowercase everything
    text = text.lower()

    # Keep: a-z, apostrophe, whitespace, and the specific diacritic
    # characters confirmed to exist in this corpus:
    # ê ñ ô ŝ š ȇ ȏ ḓ
    # Everything else (digits, punctuation like ! " ? _ , - etc.)
    # becomes a space rather than being deleted, to avoid gluing words together
    text = re.sub(r"[^a-z'êñôŝšȇȏḓ\s]", " ", text)

    # Collapse repeated whitespace and trim ends
    text = re.sub(r'\s+', ' ', text).strip()

    batch["transcript"] = text
    return batch

In [ ]:
def is_valid_transcript(batch):
    t = batch["transcript"]
    return t is not None and isinstance(t, str) and t.strip() != ""

In [ ]:
for split in ["train", "dev_test"]:
    dataset_dict[split] = dataset_dict[split].filter(is_valid_transcript, num_proc=4)
    dataset_dict[split] = dataset_dict[split].map(normalize_text)

## 6. Check whether MMS already ships an adapter for this language

If `LANG_EXISTS` is True, section 12 loads that existing adapter as your
starting point rather than initializing one from scratch — a meaningfully
better starting point since it's already been trained on real speech in a
related setup.

Note: this checks for the actual adapter *weight file*
(`adapter.<lang>.bin`) on the hub directly, not just whether a tokenizer/vocab
entry exists for the language — those are separate resources, and a vocab
entry existing does not guarantee an adapter file exists. Checking
the weight file directly is what avoids the OSError you'd otherwise only
discover during model loading.

In [ ]:
from huggingface_hub import hf_hub_download
from huggingface_hub.utils import EntryNotFoundError

try:
    hf_hub_download(repo_id=MODEL_ID, filename=f"adapter.{LANGUAGE}.bin")
    LANG_EXISTS = True
    print(f"'{LANGUAGE}' already has an MMS adapter — fine-tuning existing adapter.")
except EntryNotFoundError:
    LANG_EXISTS = False
    print(f"'{LANGUAGE}' has no shipped MMS adapter file — initializing a new adapter from scratch.")
except Exception as e:
    LANG_EXISTS = False
    print(f"Could not confirm adapter for '{LANGUAGE}' — treating as new adapter. (detail: {e})")

## 7. Build vocabulary from your transcripts

Same char-level vocab build as your wav2vec2 notebook, but saved in MMS's nested `{lang: vocab}` format. Even if `LANG_EXISTS` is True, build this — compare against MMS's shipped vocab for your language before deciding which tokenizer to use in the next section.

In [ ]:
def extract_chars(batch):
    all_text = " ".join(batch["transcript"])
    return {"vocab": [list(set(all_text))]}

vocab_set = set()
for split in ["train", "dev_test"]:
    v = dataset_dict[split].map(
        extract_chars, batched=True, batch_size=-1,
        keep_in_memory=True, remove_columns=dataset_dict[split].column_names,
    )
    for row in v["vocab"]:
        vocab_set.update(row)

vocab_dict = {v: k for k, v in enumerate(sorted(vocab_set))}
vocab_dict["|"] = vocab_dict.pop(" ", len(vocab_dict))
vocab_dict["[UNK]"] = len(vocab_dict)
vocab_dict["[PAD]"] = len(vocab_dict)

new_vocab_path = "new_vocab.json"
with open(new_vocab_path, "w", encoding="utf-8") as f:
    json.dump({LANGUAGE: vocab_dict}, f, ensure_ascii=False)

print(f"Vocab size: {len(vocab_dict)}")
vocab_dict

## 8. Build processor

In [ ]:
if LANG_EXISTS:
    processor = AutoProcessor.from_pretrained(MODEL_ID, target_lang=LANGUAGE)
    tokenizer = processor.tokenizer
else:
    tokenizer = Wav2Vec2CTCTokenizer(
        new_vocab_path,
        unk_token="[UNK]",
        pad_token="[PAD]",
        word_delimiter_token="|",
        target_lang=LANGUAGE,
    )
    feature_extractor = Wav2Vec2FeatureExtractor(
        feature_size=1, sampling_rate=16000, padding_value=0.0,
        do_normalize=True, return_attention_mask=True,
    )
    processor = Wav2Vec2Processor(feature_extractor=feature_extractor, tokenizer=tokenizer)

processor.save_pretrained(OUTPUT_DIR)

## 9. Preprocess audio + labels

Same as your wav2vec2 notebook — uses `processor.tokenizer(...)` directly rather than the deprecated `as_target_processor()` context manager.

In [ ]:
def prepare_dataset(batch):
    audio = batch["audio"]
    batch["input_values"] = processor(
        audio["array"], sampling_rate=audio["sampling_rate"]
    ).input_values[0]
    batch["input_length"] = len(batch["input_values"])
    batch["labels"] = processor.tokenizer(batch["transcript"]).input_ids
    return batch

In [ ]:
test_sentence = dataset_dict["train"][0]["transcript"]
test_sentence

In [ ]:
for split in ["train", "dev_test"]:
    dataset_dict[split] = dataset_dict[split].filter(is_valid_transcript, num_proc=2)
    dataset_dict[split] = dataset_dict[split].map(
        prepare_dataset,
        remove_columns=dataset_dict[split].column_names,
        num_proc=2,  # bump to e.g. 4-6 with 32GB RAM if you have spare cores
    )

In [ ]:
encoded = processor(text=test_sentence).input_ids
decoded = processor.decode(encoded)

print(f"Original: {test_sentence}")
print(f"Decoded:  {decoded}")

## 10. Data collator

In [ ]:
@dataclass
class DataCollatorCTCWithPadding:
    processor: Wav2Vec2Processor
    padding: Union[bool, str] = True

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_values": f["input_values"]} for f in features]
        label_features = [{"input_ids": f["labels"]} for f in features]

        batch = self.processor.pad(input_features, padding=self.padding, return_tensors="pt")
        labels_batch = self.processor.tokenizer.pad(label_features, padding=self.padding, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        batch["labels"] = labels
        return batch

data_collator = DataCollatorCTCWithPadding(processor=processor, padding=True)

## 11. Metrics (WER / CER)

Includes the `examples` field you added — this is what let you spot the collapse early in the XLSR run. Keep it here so you catch a repeat of that pattern within the first couple of eval steps.

In [ ]:
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

def compute_metrics(pred):
    pred_logits = pred.predictions
    pred_ids = np.argmax(pred_logits, axis=-1)
    pred.label_ids[pred.label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.batch_decode(pred_ids)
    label_str = processor.batch_decode(pred.label_ids, group_tokens=False)

    return {
        "wer": wer_metric.compute(predictions=pred_str, references=label_str),
        "cer": cer_metric.compute(predictions=pred_str, references=label_str),
        "examples": {"prediction": pred_str[:3], "label": label_str[:3]}
    }

## 12. Load model and set up adapter training

Key difference from your XLSR notebook: instead of unfreezing the whole
encoder (minus feature extractor), everything is frozen except the
target-language adapter + `lm_head`. This is the main structural lever
against collapse here — there simply isn't enough free capacity for the
model to cheaply learn "always predict blank/one character" the way a
fully-unfrozen 300M-parameter encoder can in a few dozen steps on a tiny
dataset.

`ctc_zero_infinity=True` and the frozen-base setup are both carried over
from what worked (or was trying to work) in your XLSR run.

**Important:** `target_lang` is deliberately *not* passed to `from_pretrained()`
here. Doing so makes `transformers` auto-load that language's adapter
internally during `tie_weights()` — before this notebook's own `LANG_EXISTS`
branch runs — and it fails with an uncatchable `OSError` if the adapter
file doesn't exist, regardless of what your own logic decides afterward.
Loading the base model with `target_lang=None`, then manually calling
`load_adapter`/`init_adapter_layers` ourselves below, is what makes the
`LANG_EXISTS` branch actually take effect.

In [ ]:
model = Wav2Vec2ForCTC.from_pretrained(
    MODEL_ID,
    target_lang=None,  # deliberately omitted here — see markdown above
    ignore_mismatched_sizes=True,
    ctc_loss_reduction="mean",
    ctc_zero_infinity=True,
    vocab_size=len(tokenizer),
)

if LANG_EXISTS:
    try:
        model.load_adapter(LANGUAGE, force_load=True)
    except Exception as e:
        print(f"Existing adapter failed to load ({e}) — falling back to a fresh adapter.")
        model.init_adapter_layers()
else:
    model.init_adapter_layers()

model.config.target_lang = LANGUAGE

# Freeze the multilingual backbone; only adapter + lm_head stay trainable.
model.freeze_base_model()
adapter_weights = model._get_adapters()
for param in adapter_weights.values():
    param.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

model = model.to("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
print("ctc_zero_infinity:", model.config.ctc_zero_infinity)
print("ctc_loss_reduction:", model.config.ctc_loss_reduction)

## 13. Training arguments

Same small-scale testing setup as your wav2vec2 run (`save_strategy="no"`,
tiny batch, high grad accumulation, low epoch count) so you can validate
the pipeline on a small subset before committing to a full run. Adapter
training tolerates a higher learning rate than full-encoder fine-tuning —
`5e-5` (your XLSR value) is on the low side for adapter-only training,
so this defaults higher; drop it back down if you see instability.

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    save_strategy="no",   # disables checkpoint saving entirely, turn on for prod
    load_best_model_at_end=False,  # disables loading best model at end, turn on for prod
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH,
    per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    eval_strategy="steps",
    eval_steps=10,
    save_steps=600,
    logging_steps=10,
    learning_rate=3e-4,  # higher than full fine-tuning is appropriate for adapter-only
    warmup_steps=50,
    num_train_epochs=3,
    fp16=torch.cuda.is_available(),
    max_grad_norm=1.0,
    gradient_checkpointing=True,
    save_total_limit=2,
    metric_for_best_model="wer",
    greater_is_better=False,
    push_to_hub=False,
    report_to=[],
)

trainer = Trainer(
    model=model,
    data_collator=data_collator,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=dataset_dict["train"],
    eval_dataset=dataset_dict["dev_test"],
    processing_class=processor.feature_extractor,
)

## 14. Train

Watch the `examples` column in the eval logs closely for the first few
evals. If predictions collapse to a single repeated character the way the
XLSR run did, stop early — with the backbone frozen this points to the
learning rate or the adapter init rather than encoder capacity, so it's
worth halving `learning_rate` and retrying before scaling up the dataset.

In [ ]:
trainer.train()

In [ ]:
trainer.state.log_history

## 15. Save

In [ ]:
# trainer.save_model(OUTPUT_DIR)
# processor.save_pretrained(OUTPUT_DIR)
# print(f"Saved to {OUTPUT_DIR}")

## 16. Quick sanity-check inference

In [ ]:
import soundfile as sf

device = "cuda" if torch.cuda.is_available() else "cpu"
model.eval()
model.to(device)

In [ ]:
sample = dataset_dict["dev_test"].select(range(1))[0]
input_values = torch.tensor(sample["input_values"]).unsqueeze(0).to(device)

with torch.no_grad():
    logits = model(input_values).logits

predicted_ids = torch.argmax(logits, dim=-1)
print("Raw predicted IDs:", predicted_ids[0].tolist())
print("Unique IDs predicted:", set(predicted_ids[0].tolist()))
print("Pad/blank token ID:", processor.tokenizer.pad_token_id)

# Test 1: preprocessed dev_test split

In [ ]:
def test_on_eval_set(num_samples=5):
    print("=== Evaluation on dev_test split ===\n")
    test_samples = dataset_dict["dev_test"].select(range(num_samples))

    for i, sample in enumerate(test_samples):
        input_values = torch.tensor(sample["input_values"]).unsqueeze(0).to(device)

        with torch.no_grad():
            logits = model(input_values).logits

        predicted_ids = torch.argmax(logits, dim=-1)
        predicted_text = processor.tokenizer.decode(predicted_ids[0])
        actual_text = processor.tokenizer.decode(sample["labels"], group_tokens=False)

        print(f"--- Sample {i+1} ---")
        print(f"Predicted: {predicted_text}")
        print(f"Actual:    {actual_text}\n")

In [ ]:
test_on_eval_set(num_samples=5)

# Test 2: raw audio files (informal validation)

In [ ]:
def transcribe_audio_file(filepath):
    audio, sr = sf.read(filepath)
    inputs = processor(audio, sampling_rate=sr, return_tensors="pt").input_values.to(device)

    with torch.no_grad():
        logits = model(inputs).logits

    predicted_ids = torch.argmax(logits, dim=-1)
    return processor.tokenizer.decode(predicted_ids[0])

In [ ]:
def test_on_raw_files(filepaths):
    print("=== Transcription on raw audio files ===\n")
    for path in filepaths:
        prediction = transcribe_audio_file(path)
        print(f"File: {path}")
        print(f"Predicted: {prediction}\n")

In [ ]:
raw_files = [
    "/home/khotso/data/validation_clips/clip1.wav",
    "/home/khotso/data/validation_clips/clip2.wav",
]
# test_on_raw_files(raw_files)